# Import Dependencies
Import TensorFlow/Keras and any required utilities for model definition and saving.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from keras.datasets import mnist
from keras.utils import to_categorical
from tensorflow import keras

# Define Camera CNN Architecture
Define the camera model architecture with the same layers, shapes, and activation functions as the existing camera model.

In [ ]:
def build_camera_model() -> keras.Model:
    return keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(32, (5, 5), activation="relu", padding="same", name="conv2d_12"),
        keras.layers.Conv2D(64, (5, 5), activation="relu", padding="same", name="conv2d_13"),
        keras.layers.MaxPooling2D((2, 2), name="max_pooling2d_9"),
        keras.layers.BatchNormalization(name="batch_normalization_5"),
        keras.layers.Conv2D(128, (5, 5), activation="relu", padding="same", name="conv2d_14"),
        keras.layers.MaxPooling2D((2, 2), name="max_pooling2d_10"),
        keras.layers.Conv2D(256, (5, 5), activation="relu", padding="same", name="conv2d_15"),
        keras.layers.MaxPooling2D((2, 2), name="max_pooling2d_11"),
        keras.layers.BatchNormalization(name="batch_normalization_6"),
        keras.layers.Dropout(0.25, name="dropout_6"),
        keras.layers.Flatten(name="flatten_4"),
        keras.layers.Dense(128, activation="relu", name="dense_7"),
        keras.layers.Dropout(0.5, name="dropout_7"),
        keras.layers.Dense(10, activation="softmax", name="dense_8"),
    ])

# Build and Summarize Camera Model
Instantiate the model and call summary() to verify layer order and output shapes.

In [ ]:
camera_model = build_camera_model()
camera_model.summary()

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1)).astype("float32") / 255
train_labels = to_categorical(train_labels)

test_images = test_images.reshape((10000, 28, 28, 1)).astype("float32") / 255
test_labels = to_categorical(test_labels)

camera_model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])

history = camera_model.fit(
    train_images,
    train_labels,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

test_loss, test_acc = camera_model.evaluate(test_images, test_labels, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Train Camera Model (MNIST)
Load MNIST, train the camera CNN, and report accuracy.

In [ ]:
if "history" in globals():
    epochs = range(1, len(history.history["accuracy"]) + 1)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history["accuracy"], label="train")
    plt.plot(epochs, history.history.get("val_accuracy", []), label="val")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history["loss"], label="train")
    plt.plot(epochs, history.history.get("val_loss", []), label="val")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No training history found. Run a training cell to generate plots.")

# Training Curves
Plot accuracy and loss over epochs (requires a training history).

In [ ]:
layer_names = [layer.name for layer in camera_model.layers]
layer_params = [layer.count_params() for layer in camera_model.layers]

print(f"Total parameters: {camera_model.count_params():,}")

plt.figure(figsize=(10, 4))
plt.bar(layer_names, layer_params)
plt.title("Camera Model Parameters by Layer")
plt.xlabel("Layer")
plt.ylabel("Params")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Model Metrics and Chart
Show parameter counts by layer for a quick architecture check.

# Export Camera Model
Save the camera model to a Keras-compatible file for later training or loading.

In [ ]:
# Colab-friendly export path
if Path("/content").exists():
    output_path = Path("/content") / "models" / "camera_model.keras"
else:
    backend_dir = Path.cwd().parent if Path.cwd().name == "model_training" else Path.cwd()
    output_path = backend_dir / "models" / "camera_model.keras"

output_path.parent.mkdir(parents=True, exist_ok=True)
camera_model.save(output_path.as_posix())
print(f"Saved camera model to: {output_path}")